In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm

# CONFIG - IDEA 2: Two Models + 90/10 Split (Same Seed)

In [ ]:
SEED = 42
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 10  # Keep epochs at 10
MODEL_PATH_BASE = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
MODEL_PATH_SMALL = "/kaggle/input/deberta-v3-small/transformers/default/1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"IDEA 2: Two models (base + small) with 90/10 split, {EPOCHS} epochs each")

In [ ]:
# Set seeds
import random
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [ ]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [ ]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [ ]:
test_df = pd.read_csv(test_path)

# Get augmented data from both df and test_df
augmented_train = add_data(df)
augmented_test = add_data(test_df)

# Combine original df with augmented data
augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]

# Create new augmented dataframe
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',  # Keep the original case of the first occurrence
    'label': 'mean'   # Take mean of labels
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

augmented_df.head()

In [ ]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

In [ ]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
def validate(model, loader):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]  # Assuming this is already on CPU as integers
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    # Convert to numpy arrays
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    # Compute AUC per rule
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        # Only compute AUC if we have both positive and negative samples for this rule
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            # If only one class present, we can't compute AUC
            rule_aucs[rule_id] = np.nan
    
    # Compute average AUC across rules (excluding NaN values)
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds

In [ ]:
from transformers import get_linear_schedule_with_warmup

In [ ]:
def train_single_model(model_path, model_name, train_data, val_data):
    """Train a single model and return best AUC"""
    print(f"\n=== Training {model_name} ===")
    
    # Load tokenizer for this model
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
    
    # Create datasets
    train_ds = JigsawDataset(
        train_data['text'].tolist(), 
        train_data['label'].tolist(), 
        train_data['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    val_ds = JigsawDataset(
        val_data['text'].tolist(), 
        val_data['label'].tolist(), 
        val_data['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    # Initialize model
    model = JigsawModel(model_path).to(DEVICE)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    
    print(f'Trainable Params: {sum(i.numel() for i in model.parameters() if i.requires_grad)}')
    
    # Setup optimizer and scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    # Training loop
    best_auc = 0
    for epoch in range(EPOCHS):
        print(f"Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        val_auc, val_loss, val_preds = validate(model, val_loader)
        
        print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            torch.save(model.state_dict(), f"model_{model_name}_best.bin")
    
    print(f"{model_name} best validation AUC: {best_auc:.4f}")
    return best_auc, tokenizer

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # IDEA 2: Two models with same 90/10 split
    print("=== IDEA 2: Training two models with 90/10 split ===")
    
    # Create single 90/10 split (same seed for both models)
    train_data, val_data = train_test_split(
        augmented_df, 
        test_size=0.1, 
        stratify=augmented_df["rule"], 
        random_state=SEED
    )
    
    print(f"Train size: {len(train_data)} (90%)")
    print(f"Val size: {len(val_data)} (10%)")
    
    # Train Model 1: DeBERTa-base
    auc1, tokenizer_base = train_single_model(
        MODEL_PATH_BASE, "deberta_base", train_data, val_data
    )
    
    # Clean up memory
    torch.cuda.empty_cache()
    
    # Train Model 2: DeBERTa-small  
    auc2, tokenizer_small = train_single_model(
        MODEL_PATH_SMALL, "deberta_small", train_data, val_data
    )
    
    print(f"\n=== IDEA 2 Results ===")
    print(f"DeBERTa-base AUC: {auc1:.4f}")
    print(f"DeBERTa-small AUC: {auc2:.4f}")
    print(f"Average AUC: {(auc1 + auc2) / 2:.4f}")

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Generate test predictions - ensemble of both models
    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    all_test_preds = []
    
    # Model 1: DeBERTa-base predictions
    model1 = JigsawModel(MODEL_PATH_BASE).to(DEVICE)
    model1.load_state_dict(torch.load("model_deberta_base_best.bin", map_location=DEVICE))
    model1.eval()
    
    test_ds1 = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), [0]*len(df_test), tokenizer_base, MAX_LEN)
    test_loader1 = DataLoader(test_ds1, batch_size=BATCH_SIZE)
    
    test_preds1 = []
    with torch.no_grad():
        for batch in tqdm(test_loader1, desc="DeBERTa-base predictions"):
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = model1(ids, mask)
            test_preds1.extend(torch.sigmoid(logits).cpu().numpy())
    
    all_test_preds.append(test_preds1)
    del model1
    torch.cuda.empty_cache()
    
    # Model 2: DeBERTa-small predictions
    model2 = JigsawModel(MODEL_PATH_SMALL).to(DEVICE)
    model2.load_state_dict(torch.load("model_deberta_small_best.bin", map_location=DEVICE))
    model2.eval()
    
    test_ds2 = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), [0]*len(df_test), tokenizer_small, MAX_LEN)
    test_loader2 = DataLoader(test_ds2, batch_size=BATCH_SIZE)
    
    test_preds2 = []
    with torch.no_grad():
        for batch in tqdm(test_loader2, desc="DeBERTa-small predictions"):
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            logits = model2(ids, mask)
            test_preds2.extend(torch.sigmoid(logits).cpu().numpy())
    
    all_test_preds.append(test_preds2)
    
    # Ensemble predictions (simple average)
    final_test_preds = np.mean(all_test_preds, axis=0)

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sample["rule_violation"] = final_test_preds
    sample.to_csv("submission_idea2.csv", index=False)
    print("✅ IDEA 2 Submission saved as submission_idea2.csv")
else:
    !touch submission_idea2.csv
    
!head -n 4 submission_idea2.csv